# 6 — RNN training (3 versions)

In this notebook, we train **three simple RNN models** to predict possible medical conditions from symptom text. Then we compare their results and choose which model to keep.

We use the same `train.json` and `validation.json` as our Logistic Regression model. **We do not use the test set here.** The dataset is made from NHS symptom descriptions, not real patient cases.

## 1. Libraries and settings

We use PyTorch for the RNN, and scikit-learn for our evaluation metrics. PyTorch will use the GPU if CUDA is available.

If necessary, install missing libraries in your notebook environment using `%pip install numpy pandas matplotlib scikit-learn torch`. For an NVIDIA GPU, install a CUDA-enabled PyTorch build from the official PyTorch website.

In [1]:
import copy
import hashlib
import json
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 64
MAX_LENGTH = 96
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.001
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Find the project folder even if VS Code starts in app/training.
ROOT = Path.cwd().resolve()
while not (ROOT / 'app' / 'dataset').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open the notebook from your medical-diagnosis project')
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'app' / 'dataset' / 'output' / 'training'
MODEL_DIR = ROOT / 'app' / 'model' / 'rnn'
REPORT_DIR = DATA_DIR / 'rnn'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', ROOT)

ModuleNotFoundError: No module named 'torch'

## 2. Load the dataset

Each example has symptom text and a list of condition labels. Because one example can have more than one label, this is a **multi-label classification** problem.

In [ ]:
train_file = DATA_DIR / 'train.json'
val_file = DATA_DIR / 'validation.json'
train_rows = json.loads(train_file.read_text(encoding='utf-8'))
val_rows = json.loads(val_file.read_text(encoding='utf-8'))

train_texts = [row['symptoms_classical'] for row in train_rows]
val_texts = [row['symptoms_classical'] for row in val_rows]

assert set(train_texts).isdisjoint(val_texts), 'Training and validation texts overlap'

label_binarizer = MultiLabelBinarizer()
y_train = label_binarizer.fit_transform([row['labels'] for row in train_rows])
y_val = label_binarizer.transform([row['labels'] for row in val_rows])
class_names = label_binarizer.classes_.tolist()
supported = y_val.sum(axis=0) > 0

# Keep a record of which dataset we trained on.
data_hashes = {
    'train_json_sha256': hashlib.sha256(train_file.read_bytes()).hexdigest(),
    'validation_json_sha256': hashlib.sha256(val_file.read_bytes()).hexdigest(),
}

print('Training examples:', len(train_rows))
print('Validation examples:', len(val_rows))
print('Conditions:', len(class_names))
print('Conditions in validation:', int(supported.sum()))
unknown_labels = set(label for r in val_rows for label in r['labels']) - set(class_names)
assert not unknown_labels, f'Unknown validation labels: {unknown_labels}'
print('Test set: not used')

## 3. Turn text into numbers

An RNN reads sequences of word IDs. We build the vocabulary only from the training data, then pad each sentence to the same length.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?|\d+", text.lower())

word_counts = Counter(word for text in train_texts for word in tokenize(text))
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common(24998):
    vocab[word] = len(vocab)

def encode(text):
    words = tokenize(text)[:MAX_LENGTH]
    ids = [vocab.get(word, 1) for word in words] or [1]
    return ids + [0] * (MAX_LENGTH - len(ids))

train_x = torch.tensor([encode(text) for text in train_texts], dtype=torch.long)
val_x = torch.tensor([encode(text) for text in val_texts], dtype=torch.long)
train_y = torch.tensor(y_train, dtype=torch.float32)
val_y = torch.tensor(y_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=BATCH_SIZE)
print('Vocabulary size:', len(vocab))

## 4. Our RNN model

Our network uses an embedding layer, a **bidirectional vanilla RNN**, and a linear output layer. We use sigmoid scores because each condition is predicted independently. This is an RNN, **not an LSTM**.

In [ ]:
class SymptomRNN(nn.Module):
    def __init__(self, vocab_size, num_classes, hidden_size=128, layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.rnn = nn.RNN(128, hidden_size, num_layers=layers,
                          dropout=dropout if layers > 1 else 0,
                          bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).cpu()
        words = self.embedding(x)
        packed = pack_padded_sequence(words, lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(last_hidden))

# Each version changes the RNN size or number of layers.
versions = {
    'v1': {'hidden_size': 128, 'layers': 1, 'dropout': 0.30},
    'v2': {'hidden_size': 192, 'layers': 1, 'dropout': 0.30},
    'v3': {'hidden_size': 128, 'layers': 2, 'dropout': 0.40},
}
display(pd.DataFrame(versions).T)

## 5. Train each version

We train using weighted binary cross-entropy, stop early if validation loss is not improving, and keep the weights from the best epoch. We do not change the validation examples between versions.

In [ ]:
# Give extra importance to conditions that appear less often.
positive = y_train.sum(axis=0)
class_weights = np.clip((len(train_rows) - positive) / np.maximum(positive, 1), 1, 15)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))

def train_model(settings):
    torch.manual_seed(SEED)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(SEED)

    model = SymptomRNN(len(vocab), len(class_names), **settings).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
    history = []
    best_loss = float('inf')
    best_epoch = 0
    waiting = 0
    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(x)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                val_loss += loss_fn(model(x), y).item() * len(x)

        train_loss /= len(train_rows)
        val_loss /= len(val_rows)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': val_loss})
        print(f'  Epoch {epoch:02d}: train={train_loss:.4f}, val={val_loss:.4f}')

        if val_loss < best_loss - 0.0001:
            best_loss = val_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            waiting = 0
        else:
            waiting += 1
            if waiting == PATIENCE:
                print('  Early stopping')
                break

    model.load_state_dict(best_weights)
    return model, pd.DataFrame(history), best_epoch, time.perf_counter() - start

## 6. Evaluate each model

We choose a threshold using **validation Micro F1** and also check the Top-1, Top-3, and Top-5 ranked conditions. A Top-K hit means at least one correct label appears in the first K suggestions.

In [ ]:
def evaluate_model(model):
    model.eval()
    scores = []
    with torch.no_grad():
        for x, _ in val_loader:
            predictions = torch.sigmoid(model(x.to(DEVICE)))
            scores.append(predictions.cpu().numpy())
    scores = np.concatenate(scores)

    thresholds = [0.005, 0.01, 0.02, 0.025] + list(np.round(np.arange(0.05, 0.951, 0.025), 3))
    threshold_rows = []
    for threshold in thresholds:
        predicted = (scores >= threshold).astype(int)
        threshold_rows.append({
            'threshold': float(threshold),
            'micro_precision': float(precision_score(y_val, predicted, average='micro', zero_division=0)),
            'micro_recall': float(recall_score(y_val, predicted, average='micro', zero_division=0)),
            'micro_f1': float(f1_score(y_val, predicted, average='micro', zero_division=0)),
        })

    best = max(threshold_rows, key=lambda row: row['micro_f1'])
    predicted = (scores >= best['threshold']).astype(int)

    top_k = {}
    for k in (1, 3, 5):
        indexes = np.argsort(-scores, axis=1)[:, :k]
        found = np.take_along_axis(y_val, indexes, axis=1)
        top_k[f'top_{k}'] = {
            'top_k': k,
            'hit_rate': float(np.mean(found.sum(axis=1) > 0)),
            'label_recall_at_k': float(np.mean(found.sum(axis=1) / y_val.sum(axis=1))),
        }

    metrics = {
        'selected_threshold': best['threshold'],
        'micro_precision': best['micro_precision'],
        'micro_recall': best['micro_recall'],
        'micro_f1': best['micro_f1'],
        'macro_f1_all_classes': float(f1_score(y_val, predicted, average='macro', zero_division=0)),
        'macro_f1_validation_supported_classes': float(f1_score(y_val[:, supported], predicted[:, supported], average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_val, predicted, average='weighted', zero_division=0)),
        'top_k': top_k,
    }
    return metrics, threshold_rows

## 7. Train, evaluate and save V1–V3

Each version saves a `.pt` model, a validation-metrics JSON, and a CSV of its training history. The files go directly into the `rnn` folders, with **no V1/V2/V3 subfolders**.

If you have previously saved `rnn_v1.pt`, move or rename the old file before running this notebook. This notebook will not overwrite existing model files.

In [ ]:
# Check output names before training so we do not accidentally overwrite anything.
for version in versions:
    name = f'rnn_{version}'
    for path in [MODEL_DIR / f'{name}.pt',
                 REPORT_DIR / f'{name}_validation_metrics.json',
                 REPORT_DIR / f'{name}_training_history.csv']:
        if path.exists():
            raise FileExistsError(f'File already exists: {path}. Move it or rename it before running.')

results = []
all_history = {}

for version, settings in versions.items():
    name = f'rnn_{version}'
    print(f'\n===== Training {name} =====')
    model, history, best_epoch, seconds = train_model(settings)
    metrics, threshold_search = evaluate_model(model)

    metadata = {
        'name': name,
        'type': 'Bidirectional vanilla RNN (multi-label)',
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'training_examples': len(train_rows),
        'validation_examples': len(val_rows),
        'class_count': len(class_names),
        'validation_supported_class_count': int(supported.sum()),
        'validation_missing_class_count': int((~supported).sum()),
        'best_epoch': best_epoch,
        'trained_epochs': len(history),
        'training_seconds': round(seconds, 3),
        'pytorch_version': str(torch.__version__),
        'device_used': DEVICE,
        'random_seed': SEED,
        'dataset_sha256': data_hashes,
        'model_config': settings,
        'training': {'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
                     'early_stopping_patience': PATIENCE},
        'evaluation_scope': 'NHS-derived validation examples; test set not used',
        'probability_note': 'Model scores are not calibrated medical probabilities',
        'demographics_note': 'Age and gender are not included in the source training examples',
    }

    model_file = MODEL_DIR / f'{name}.pt'
    metrics_file = REPORT_DIR / f'{name}_validation_metrics.json'
    history_file = REPORT_DIR / f'{name}_training_history.csv'

    checkpoint = {
        'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'model_config': settings,
        'vocab': vocab,
        'class_names': class_names,
        'selected_threshold': metrics['selected_threshold'],
        'metadata': metadata,
    }
    torch.save(checkpoint, model_file)
    report = {'model': name, 'metadata': metadata,
              'validation_metrics': metrics, 'threshold_search': threshold_search,
              'model_file': str(model_file), 'test_set_used': False}
    metrics_file.write_text(json.dumps(report, indent=2), encoding='utf-8')
    history.to_csv(history_file, index=False)

    # Check that our saved model can be loaded again.
    loaded = torch.load(model_file, map_location='cpu', weights_only=True)
    check_model = SymptomRNN(len(loaded['vocab']), len(loaded['class_names']),
                             **loaded['model_config'])
    check_model.load_state_dict(loaded['model_state_dict'])
    assert loaded['class_names'] == class_names

    all_history[name] = history
    results.append({
        'model': name,
        'hidden_size': settings['hidden_size'],
        'layers': settings['layers'],
        'best_epoch': best_epoch,
        'threshold': metrics['selected_threshold'],
        'micro_precision': metrics['micro_precision'],
        'micro_recall': metrics['micro_recall'],
        'micro_f1': metrics['micro_f1'],
        'macro_f1_supported': metrics['macro_f1_validation_supported_classes'],
        'weighted_f1': metrics['weighted_f1'],
        'top_1': metrics['top_k']['top_1']['hit_rate'],
        'top_3': metrics['top_k']['top_3']['hit_rate'],
        'top_5': metrics['top_k']['top_5']['hit_rate'],
        'training_seconds': round(seconds, 2),
    })
    print(f'Saved {name} | Micro F1: {metrics["micro_f1"]:.2%} | Top-5: {metrics["top_k"]["top_5"]["hit_rate"]:.2%}')
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print('\nAll three versions finished.')

## 8. Compare the three RNN versions

We sort by Micro F1 and look at Top-K ranking too. The version that wins one metric might not win every metric.

In [ ]:
comparison = pd.DataFrame(results).sort_values('micro_f1', ascending=False)
percent_cols = ['micro_precision', 'micro_recall', 'micro_f1',
                'macro_f1_supported', 'weighted_f1', 'top_1', 'top_3', 'top_5']
display(comparison.style.format({col: '{:.2%}' for col in percent_cols}))

comparison.to_csv(REPORT_DIR / 'rnn_comparison.csv', index=False)
(REPORT_DIR / 'rnn_comparison.json').write_text(json.dumps({
    'selected_model': comparison.iloc[0]['model'],
    'selection_metric': 'validation micro_f1',
    'dataset_sha256': data_hashes,
    'comparison': results,
    'test_set_used': False,
}, indent=2), encoding='utf-8')

fig, ax = plt.subplots(figsize=(8, 4))
comparison.set_index('model')[['micro_f1', 'top_1', 'top_3', 'top_5']].mul(100).plot.bar(ax=ax)
ax.set_ylabel('Validation score (%)')
ax.set_title('RNN versions: validation comparison')
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Training loss graphs

We also plot the training and validation loss for each version to see if the model is improving or starting to overfit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, history) in zip(axes, all_history.items()):
    ax.plot(history['epoch'], history['train_loss'], label='Train')
    ax.plot(history['epoch'], history['validation_loss'], label='Validation')
    ax.set_title(name.upper())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
plt.tight_layout()
plt.show()

## 10. Compare with Logistic Regression

These are our earlier **validation** results for the selected Logistic Regression baselines, not results from running that model here. They use the same frozen train/validation split.

In [ ]:
best_f1 = comparison.iloc[0]
best_top5 = comparison.sort_values('top_5', ascending=False).iloc[0]

baseline = pd.DataFrame([
    {'model': 'LogReg (best F1)', 'micro_f1': 0.783640081799591,
     'top_1': 0.887000849617672, 'top_3': 0.9498725573491928, 'top_5': 0.9694137638062872},
    {'model': 'LogReg (best ranking)', 'micro_f1': 0.780448065173116,
     'top_1': 0.897196261682243, 'top_3': 0.9575191163976211, 'top_5': 0.9753610875106202},
])
combined = pd.concat([baseline, comparison[['model', 'micro_f1', 'top_1', 'top_3', 'top_5']]], ignore_index=True)
display(combined.style.format({col: '{:.2%}' for col in ['micro_f1', 'top_1', 'top_3', 'top_5']}))
print('Best RNN by Micro F1:', best_f1['model'])
print('Best RNN by Top-5:', best_top5['model'])

## 11. Try a symptom example

Load the highest-F1 RNN from disk and show its top 5 suggested conditions. This is just a quick model check, **not a medical diagnosis**.

In [ ]:
best_name = comparison.iloc[0]['model']
saved = torch.load(MODEL_DIR / f'{best_name}.pt', map_location=DEVICE, weights_only=True)
best_model = SymptomRNN(len(saved['vocab']), len(saved['class_names']),
                        **saved['model_config']).to(DEVICE)
best_model.load_state_dict(saved['model_state_dict'])
best_model.eval()

sample_text = 'persistent cough wheezing and shortness of breath'
sample = torch.tensor([encode(sample_text)], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    scores = torch.sigmoid(best_model(sample))[0].cpu().numpy()

best_indexes = np.argsort(-scores)[:5]
display(pd.DataFrame({
    'condition': np.array(class_names)[best_indexes],
    'model_score': np.round(scores[best_indexes], 4),
    'above_threshold': scores[best_indexes] >= saved['selected_threshold'],
}))

## Conclusion

We now have three RNN models and validation reports saved in our project. We can choose a candidate using the comparison table and later compare it to LSTM and BioBERT.

**Important:** Our samples come from NHS symptom descriptions, so good validation numbers do not mean the model is accurate on actual patients. The test set remains untouched. We also still need to evaluate emergency-condition recall and investigate how to handle age and gender data.